In [10]:
print("Q3: Remove duplicate rows by user_id and transaction_date")
transactions_df = superstore_numeric.select(
    col("Customer ID").alias("user_id"),
    col("Order Date").alias("transaction_date"),
    col("Order ID").alias("transaction_id")
)
transactions_df.dropDuplicates(["user_id", "transaction_date"]).show(5, truncate=False)

print("Q4: Filter West and group by product_category average sale_amount")
sales_df = superstore_numeric.select(
    col("Region").alias("region"),
    col("Category").alias("product_category"),
    col("Sales_num").alias("sale_amount")
)
(
    sales_df.filter(col("region") == "West")
    .groupBy("product_category")
    .agg(avg("sale_amount").alias("avg_sale_amount"))
).show(5, truncate=False)

print("Q5: Fill null values in status with 'Unknown'")
status_df = superstore_numeric.withColumn(
    "status",
    when(col("Profit") < 0, None).otherwise("Profitable")
)
status_df.na.fill({"status": "Unknown"}).show(5, truncate=False)

print("Q6: Total count of records per city where count > 100")
city_df = superstore_numeric.select(col("City").alias("city"))
(
    city_df.groupBy("city")
    .agg(count("*").alias("city_count"))
    .filter(col("city_count") > 100)
).show(5, truncate=False)

print("Q8: Filter West transactions with 10% to 20% discount")
filtered_df = superstore_numeric.filter(
    (col("Region") == "West") &
    (col("Discount_num") >= 0.1) &
    (col("Discount_num") <= 0.2)
)
filtered_df.show(5, truncate=False)

print("Q10: Cast Order Date to TimestampType and rename to event_time")
timestamp_df = superstore_clean.withColumn(
    "event_time",
    to_timestamp(col("Order Date"), "M/d/yyyy")
).drop("Order Date")
timestamp_df.printSchema()
timestamp_df.show(5, truncate=False)

print("Q12: Remove rows where Customer ID is null OR Ship Mode is empty")
clean_store_df = superstore_clean.filter(
    col("Customer ID").isNotNull() & (col("Ship Mode") != "")
)
clean_store_df.show(5, truncate=False)

print("Q13: Calculate min, max, and mean of Sales")
price_summary_df = superstore_numeric.agg(
    min("Sales_num").alias("min_sales"),
    max("Sales_num").alias("max_sales"),
    avg("Sales_num").alias("mean_sales")
)
price_summary_df.show(truncate=False)

print("Q15: Final processing pipeline")
final_revenue_df = (
    superstore_numeric.dropDuplicates(["Order ID"])
    .na.fill({"Sales_num": 0})
    .groupBy("Region")
    .agg(spark_sum(col("Sales_num")).alias("total_revenue"))
)
final_revenue_df.show(10, truncate=False)

spark.stop()

Q3: Remove duplicate rows by user_id and transaction_date


AssertionError: 

In [ ]:
superstore_clean = (
    superstore_df.dropDuplicates(["Order ID", "Order Date"])
    .na.fill({"Ship Mode": "Unknown"})
)

superstore_numeric = (
    superstore_clean
    .withColumn("Sales_num", expr("try_cast(Sales AS DOUBLE)"))
    .withColumn("Discount_num", expr("try_cast(Discount AS DOUBLE)"))
)

superstore_numeric.show(5, truncate=False)

+------+--------------+----------+---------+--------------+-----------+----------------+-----------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+-----------------------------------------------------+------------+--------+--------+--------+---------+------------+
|Row ID|Order ID      |Order Date|Ship Date|Ship Mode     |Customer ID|Customer Name   |Segment    |Country      |City         |State     |Postal Code|Region|Product ID     |Category       |Sub-Category|Product Name                                         |Sales       |Quantity|Discount|Profit  |Sales_num|Discount_num|
+------+--------------+----------+---------+--------------+-----------+----------------+-----------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+-----------------------------------------------------+------------+--------+--------+--------+---------+------------+
|2718  |CA-2014-100006|9/7/2014  |9/1

In [ ]:
data_path = Path.cwd() / "superstore.csv"
if not data_path.exists():
    data_path = Path.cwd().parent / "Assignment_3" / "superstore.csv"

if not data_path.exists():
    raise FileNotFoundError(f"Superstore dataset not found at {data_path}")

superstore_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(str(data_path))
)

print("=== Loaded Superstore dataset ===")
superstore_df.printSchema()
superstore_df.show(5, truncate=False)

=== Loaded Superstore dataset ===
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------

In [ ]:
spark = (
    SparkSession.builder
    .master("local[1]")
    .appName("Week5SparkAssignment")
    .config("spark.pyspark.driver.python", python_executable)
    .config("spark.pyspark.python", python_executable)
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark

In [ ]:
%pip install pyspark -q

import os
from pathlib import Path

python_executable = r"C:\Python314\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = python_executable
os.environ["PYSPARK_PYTHON"] = python_executable

try:
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import avg, col, count, expr, min, max, sum as spark_sum, when, to_timestamp
except ImportError as e:
    raise ImportError("PySpark is not installed. Install it with: python -m pip install pyspark") from e



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\AMRISH\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


# Spark Assignment Notebook

This notebook runs the Spark workflow for the Week 5 assignment using the Superstore dataset.

It includes:
- PySpark setup
- Loading the dataset
- Data cleaning and transformation
- Aggregation examples
- Sample outputs
